# 06-DEEPAGENT/test.ipynb
1. csv 파일을 받아서
2. 데이터 분석을 진행하고
3. 슬랙으로 결과를 보냄


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# Daytona Sandbox

In [2]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)



DaytonaError: Failed to create sandbox: Total disk limit exceeded. Maximum allowed: 30GiB.
Consider archiving your unused Sandboxes to free up available storage.
To increase concurrency limits, upgrade your organization's Tier by visiting https://app.daytona.io/dashboard/limits.

In [ ]:
result = backend.execute('ls -a')
print(result.output)

.
..
.bash_logout
.bashrc
.daytona
.face
.face.icon
.profile
.zshrc


In [3]:
import csv
import io

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)

# 원본 csv 데이터
csv_bytes = buf.getvalue().encode('utf-8')
buf.close()

# 파일 저장(지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드
backend.upload_files(
    [
        ('/home/daytona/data/sales_data.csv', csv_bytes)
    ]
)

NameError: name 'backend' is not defined

### Slack
- Agent가 사용할 Slack Messaging Tool 생성

In [ ]:
import os
from langchain.tools import tool
from slack_sdk import WebClient

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')
SLACK_BOT_TOKEN

SOCIAL_CHANNEL_ID = os.getenv("SOCIAL_CHANNEL_ID")

client = WebClient(token=SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()

for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = SOCIAL_CHANNEL_ID

client.chat_postMessage(
    channel = social_channel_id,
    text = "hello"
)

slack-전체 C0A3Z5N5NTX
새-채널 C0A40HEQT8E
소셜 C0A466BP3M2


In [13]:
import os
from slack_sdk import WebClient
from langchain.tools import tool

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)

@tool(parse_docstring=True) # LLM에게 Tool 설명을 docstring에서
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메시지 전송, 경우에 따라 이미지 같은 파일을 첨부함.
    
    Args:
        text: (str) 메시지의 내용
        file_path: (str) 파일시스템상 첨부파일의 경로
        
    """


    # 첨부 파일 없으면
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text=text)
    else:
        fp = backend.download_files([file_path])
        slack_client.files_upload_v2(
            channel=social_channel_id,
            content=fp[0].content,
            initial_comment=text
        )
    return 'Message sent.'

## Build Deep Agent

In [ ]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[send_slack_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

In [15]:

input_message = {
    'role': 'user',
    'content': '''
    현재 폴더 안에 ./data/sales_data.csv 파일을 분석하고, 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_057e630422eee6c30069afb015238c8193b9650ed11314d7d2', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"todos":[{"content":"Verify existence of /data/sales_data.csv","status":"in_progress"},{"content":"Read and inspect /data/sales_data.csv","status":"pending"},{"content":"Analyze data and create visualizations (save images to /tmp)","status":"pending"},{"content":"Send analysis results and visualization images to Slack","status":"pending"},{"content":"Mark all work completed and report back to user","status":"pending"}]}', 'call_id': 'call_cFvO0U49fh0OhMTpKV7iu0vw', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_057e630422eee6c30069afb02d4fec819381a0ab429acf0ea1', 'status': 'completed'}]
Tool Calls:
  write_todos (call_cFvO0U49fh0OhMTpKV7iu0vw)
 Call ID: call_cFvO0U49fh0OhMTpKV7iu0vw
  Args:
    todos: [{'content': 'Verify existence of /data/sales_data.csv', 'status': 'in_progres

In [ ]:
input_message = {
    'role': 'user',
    'content': '''
    채널은 기본 세팅 되어있음. 2번 데이터 시각화 해서 슬랙 메시지로 보내줘
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_044e08ccfaed068d0069afaa5394a081909776a5d878c770fb', 'summary': [], 'type': 'reasoning'}, {'type': 'text', 'text': '어떤 파일(또는 경로)의 "2번 데이터"를 시각화하길 원하나요? (예: /data/dataset2.csv 또는 업로드)  \n\n원하시는 시각화 종류가 있나요? 예시:\n- 단일 수치형 열: 히스토그램/박스플롯\n- 두 변수 관계: 산점도(옵션: 회귀선)\n- 범주별 비교: 막대그래프\n- 시계열: 선그래프\n- 변수들 간 상관관계: 히트맵\n지정하지 않으시면 숫자 열 기본 통계 히스토그램 + 상관행렬 히트맵(요약 이미지)을 만들어 보내겠습니다.\n\n슬랙 메시지 본문(또는 기본 채널로 보내기, 기본 텍스트 사용)을 어떻게 할까요? (예: "2번 데이터 시각화 결과입니다.") \n\n원하시면 제가 기본 옵션으로 진행하겠습니다 — 파일 경로와 시각화 선호를 알려주세요.', 'annotations': [], 'id': 'msg_044e08ccfaed068d0069afaa57ebf8819084c60b1a52e41cf0'}]
